<a href="https://colab.research.google.com/github/Narware-Ayush/AI_ML/blob/main/TransferLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


In [6]:
x=[]
y=[]
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.2,random_state=42)


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [3]:
#transformations
from torchvision.transforms import transforms
custom_transform=transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.299,0.224,0.225])
    ]
)

In [11]:
from PIL import Image
class CustomDataset(Dataset):
  def __init__(self,features,labels,transform):
    self.features=features
    self.labels=labels
    self.transform=transform

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):

    image=self.feature[index].reshape(28,28)
    image.astype(np.uint8)
    image=np.stack([image]*3,axis=-1)
    image=Image.fromarray(image)
    image=self.transform(image)
    return image , torch.tensor(self.labels[index],dtype=torch.long)


In [ ]:
train_dataset=CustomDataset(xtrain,ytrain,transform=custom_transform)
test_dataset=CustomDataset(xtest,ytest,transform=custom_transform)

train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=True,pin_memory=True)

In [5]:
#fetch the pretrained model from torch.vision
import torchvision.models as models
vgg16=models.vgg16(pretrained=True)



/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:04<00:00, 122MB/s]


In [6]:
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [7]:
for param in vgg16.features.parameters():
  param.requires_grad=False

In [8]:
from torch.nn.modules.linear import Linear
vgg16.classifier=nn.Sequential(
    nn.Linear(25088,1024),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(1024,512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512,10),

)

In [ ]:
vgg16=vgg16.to(device)

In [10]:
learning_rate=0.001
epochs=10
criterion=nn.CrossEntropyLoss()
optimizer=optim.SGD(vgg16.classifier.parameters(),lr=learning_rate)

In [ ]:
for epoch in range(epochs):
  total_epoch_loss=0
  for batch_features,batch_label in train_loader:
    batch_features,batch_label=batch_features.to(device),batch_label.to(device)
    output= vgg16(batch_features)
    loss=criterion(output,batch_label)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_epoch_loss=total_epoch_loss+loss.item()

  avg_loss=total_epoch_loss/len(train_loader)

In [12]:
vgg16.eval()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [ ]:
total=0
correct=0

with torch.no_grad():
  for batch_features,batch_label in test_loader:
    batch_features,batch_label=batch_features.to(device),batch_label.to(device)
    output=vgg16(batch_features)
    _,predicted=torch.max(output,1)
    total=total+batch_label.shape[0]
    correct=correct+(predicted==batch_label).sum().item()

  print(correct/total)
